#Step1: Load the data, Pre-process, embedding, complete RAG pipeline, and evaulate the performance of the project

In [30]:
#install all the required packages
%pip install -r requirements.txt


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [31]:
# Load the dataset
!cd scripts && python 01_load_data.py


[1/1] Loading synthetic dataset files...
  Loaded jira_tickets.json            → 300 records
  Loaded jenkins_logs.json            → 150 records
  Loaded duplicate_pairs.json         → 132 records
  Loaded ground_truth_labels.json     → 150 records

📊 Sanity check — Jira tickets
  Category distribution : Counter({'automation': 103, 'product': 102, 'infrastructure': 95})
  Status distribution   : Counter({"Won't Fix": 41, 'Resolved': 41, 'Duplicate': 35, 'Open': 33, 'Fixed': 33, 'Closed': 32, 'Reopened': 30, 'Under Investigation': 29, 'In Progress': 26})
  Sample ticket_id      : PSTR-1000
  Sample summary        : Pipeline fails at checkout stage due to Git credential expiry ...

📊 Sanity check — Jenkins logs
  Category distribution : Counter({'product': 57, 'infrastructure': 53, 'automation': 40})
  Sample build_id        : BUILD-5000
  Sample failure_summary : Test data cleanup script fails leaving orphaned volumes on fc-matrix-env ...

✅ Data loading complete. Proceed to 02_preproc

In [32]:
#Preprocess the dataset
!cd scripts && python 02_preprocess.py


[1/1] Loading synthetic dataset files...
  Loaded jira_tickets.json            → 300 records
  Loaded jenkins_logs.json            → 150 records
  Loaded duplicate_pairs.json         → 132 records
  Loaded ground_truth_labels.json     → 150 records

[1/2] Preprocessing Jira tickets (concat title + description + resolution)...
  ✓ 300 Jira tickets preprocessed
  Sample text_blob (first 200 chars):
  Pipeline fails at checkout stage due to Git credential expiry . Jenkins pipeline on powerstore-lab-02 failing at source checkout stage. Git credentials token expired 3 days ago. 6 pipelines affected. ...

[2/2] Preprocessing Jenkins logs (ANSI strip + stage chunk + error extract)...
  ✓ 150 Jenkins logs preprocessed
  Sample cleaned_text (first 300 chars):
  Failure summary: Test data cleanup script fails leaving orphaned volumes on fc-matrix-env
Component: REST-API
Error: Cleanup failed: DELETE /api/rest/v1/volume/VOL89DLBNP7 returned HTTP 401 Unauthorized. Token expired during long-runnin

In [33]:
#embed the data with BAAI/bge-large-en-v1.5 embedding model
!cd scripts && python 03_embed_index.py


[1/5] Loading dense embedding model: BAAI/bge-large-en-v1.5 ...
Loading weights: 100%|████████████████████| 391/391 [00:00<00:00, 11054.53it/s]
/Users/harisha/Downloads/final_project_post_evaluation/bug_triage_pipeline/scripts/03_embed_index.py:77: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"  ✓ Model loaded in {time.time()-t0:.1f}s  (output dim = {model.get_sentence_embedding_dimension()})")
  ✓ Model loaded in 8.3s  (output dim = 1024)

[2/5] Loading sparse BM25 model: Qdrant/bm25 ...
  ✓ BM25 model loaded in 0.1s

[3/5] Creating HYBRID Qdrant collection 'jira_bug_tickets' (dense dim=1024 cosine + sparse BM25)...
  (existing collection deleted — rebuilding fresh)
  ✓ Hybrid collection 'jira_bug_tickets' created (named vectors: 'dense', 'bm25_sparse')

[4/5] Embedding 300 Jira ticket text blobs (dense BGE-large + sparse BM25)...
Batches: 100%|█████████████████████████████████| 10/10 [00:10<00:00,  1.02s/it]
  ✓

In [34]:
# RAG pipeline
!cd scripts && python 04_rag_pipeline.py

Checking Ollama connection and qwen3:8b availability...
  ✓ qwen3:8b is available

Loading hybrid retrieval models (dense BGE-large + sparse BM25)...

[1/5] Loading dense embedding model: BAAI/bge-large-en-v1.5 ...
Loading weights: 100%|████████████████████| 391/391 [00:00<00:00, 12426.39it/s]
/Users/harisha/Downloads/final_project_post_evaluation/bug_triage_pipeline/scripts/03_embed_index.py:77: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"  ✓ Model loaded in {time.time()-t0:.1f}s  (output dim = {model.get_sentence_embedding_dimension()})")
  ✓ Model loaded in 7.2s  (output dim = 1024)

[2/5] Loading sparse BM25 model: Qdrant/bm25 ...
  ✓ BM25 model loaded in 0.0s

Connecting to Qdrant collection...

Running full triage pipeline on 5 sample failures...

[BUILD-5000] action=NEW_ISSUE  match=None  score=0.833  (57.1s)
[BUILD-5001] action=NEW_ISSUE  match=None  score=0.500  (42.1s)
[BUILD-5002] action=WORKAROUND_AVA

In [35]:
# evaluate the RAG pipeline
!cd scripts && python 05_evaluate.py

Running evaluation in 'production' mode...
(use --demo if BAAI/bge-large-en-v1.5 or Qwen3-8B/Ollama aren't available)
Loaded 150 ground truth labels

EXPERIMENT 1 — Cosine similarity threshold sweep

[1/5] Loading dense embedding model: BAAI/bge-large-en-v1.5 ...
Loading weights: 100%|████████████████████| 391/391 [00:00<00:00, 10519.46it/s]
/Users/harisha/Downloads/final_project_post_evaluation/bug_triage_pipeline/scripts/03_embed_index.py:77: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"  ✓ Model loaded in {time.time()-t0:.1f}s  (output dim = {model.get_sentence_embedding_dimension()})")
  ✓ Model loaded in 9.3s  (output dim = 1024)

[2/5] Loading sparse BM25 model: Qdrant/bm25 ...
  ✓ BM25 model loaded in 0.0s
Traceback (most recent call last):
  File "/Users/harisha/Downloads/final_project_post_evaluation/bug_triage_pipeline/scripts/05_evaluate.py", line 506, in <module>
    main()
    ~~~~^^
  File "/Users/ha